# Chapter 06 — RAG pipelines · Homework (self-test)

**Goal:** check that you can decide when RAG is needed, pick chunking and indexing strategies, fuse and measure retrieval, and build cited, cache-friendly requests.

- Theory: [README.md](README.md) · Practice: [01_practice.ipynb](01_practice.ipynb)
- Estimated time: 60–90 minutes
- Estimated API cost: **$0**. Everything graded runs offline; nothing here calls an API.
- Prerequisites: the practice notebook, and the venv from [00-prerequisites](../../00-prerequisites/README.md)

**How to use it**
1. Attempt everything **without peeking**. Solutions are at the very bottom, collapsed.
2. Part A: answer 8 scenario questions in the `answers` cell, then run the grader.
3. Part B: replace each `TODO` stub, then run its checker. Checkers never crash the notebook; they tell you what is wrong.
4. Part C: write a short design answer, then compare it with the rubric.
5. Run the **Scorecard**. Pass mark: **72%** overall (quiz + exercises).

In [ ]:
import hashlib
import json
import re
from dataclasses import dataclass, field
from typing import Callable

import anthropic
import httpx2

RESULTS: dict[int, bool] = {}
SCORE: dict[str, int] = {}


def _run_check(n: int, check: Callable[[], None]) -> None:
    """Run one checker; never let it crash the notebook."""
    try:
        check()
        RESULTS[n] = True
        print(f"✓ Exercise {n} passed")
    except NotImplementedError:
        RESULTS[n] = False
        print(f"✗ Exercise {n}: not implemented yet (the stub still raises NotImplementedError)")
    except AssertionError as exc:
        RESULTS[n] = False
        print(f"✗ Exercise {n}: {exc}")
    except Exception as exc:  # noqa: BLE001 - report, don't crash
        RESULTS[n] = False
        print(f"✗ Exercise {n}: your code raised {type(exc).__name__}: {str(exc)[:200]}")


@dataclass
class Chunk:
    id: str
    doc_id: str
    text: str
    meta: dict = field(default_factory=dict)


def fake_claude(seen: list) -> anthropic.Anthropic:
    """A real Anthropic client whose HTTP layer records the body and returns a minimal reply."""
    def handle(request: httpx2.Request) -> httpx2.Response:
        seen.append(json.loads(request.content))
        return httpx2.Response(200, json={"id": "msg_1", "type": "message", "role": "assistant", "model": "fake",
                                          "content": [{"type": "text", "text": "ok"}], "stop_reason": "end_turn",
                                          "stop_sequence": None, "usage": {"input_tokens": 1, "output_tokens": 1}})
    return anthropic.Anthropic(api_key="offline", max_retries=0,
                               http_client=anthropic.DefaultHttpxClient(transport=httpx2.MockTransport(handle)))


print("anthropic", anthropic.__version__, "| helpers ready")

## Part A — Quiz (8 questions)

Pick one letter per question.

**Q1. (Solution design)** A 300-person company wants an assistant that answers questions about its employee handbook: about 70,000 tokens, updated twice a year, the same for every employee. A teammate has started building a vector database pipeline. What do you recommend?

- A) Finish the vector pipeline, because RAG is always cheaper than a long prompt
- B) Fine-tune a model on the handbook so it needs no context at all
- C) Put the whole handbook in the prompt and use prompt caching; revisit RAG only if the corpus grows or access must differ per user
- D) Split the handbook into 50-token chunks and send the top 3 for each question to keep prompts small

**Q2. (Structured data extraction)** A legal team's assistant must cite the exact clause behind every answer. Contracts are chunked into fixed 800-token windows, and citations often point to a window that starts halfway through one clause and ends inside the next. What is the best change?

- A) Chunk on clause and section boundaries, keep the heading path (contract, section, clause number) in metadata, and split only very long clauses
- B) Increase the window to 2,000 tokens so each window holds whole clauses more often
- C) Keep the windows but tell the model to quote only complete clauses
- D) Reduce overlap to zero so windows never repeat text

**Q3. (Customer support assistant)** A support bot for a kitchen-appliance brand answers "do you sell espresso machines?" with confident details taken from a coffee-grinder manual. The brand sells no espresso machines. Logs show that for questions like this, the best reranker score is far below the scores seen on answerable questions. What should you add?

- A) Raise top-k from 5 to 20 so the model sees more context before it answers
- B) A system-prompt rule: "Only answer questions about products we sell"
- C) A larger generation model that is better at noticing irrelevant sources
- D) A gate in code: when the best reranker score is below a threshold set on labeled answerable and unanswerable questions, return "not covered" without calling the model, and keep the "answer only from the sources" instruction as a second check

**Q4. (Solution design)** You plan contextual retrieval for 40,000 reports of about 10,000 tokens each, chunked into about 12 chunks per report. Finance worries about the cost of sending every report once per chunk. Which plan keeps the cost down without dropping the technique?

- A) Generate the context with the largest model so that it needs only one pass
- B) Put the report first with `cache_control` so chunks 2–12 read it from the cache, use a fast model such as Claude Haiku 4.5, run it through the Message Batches API, and re-run only for changed reports
- C) Send only the chunk (without the report) to the contextualizer
- D) Skip contextualizing BM25 and contextualize only the embeddings

**Q5. (Enterprise RAG)** An insurer's assistant handles "What does the Basic plan cover for dental?" and "What does the Plus plan cover for dental?" well. But "Compare dental cover in the Basic and Plus plans" returns three chunks from the Plus brochure and none from Basic, so the answer is one-sided. What change fits best?

- A) Raise the chunk size so both plans' dental sections land in one chunk
- B) Use HyDE (embed a hypothetical answer) for every question
- C) Decompose the question into one sub-question per plan, retrieve for each (with a `plan` metadata filter where available), and pass both result sets to the generator
- D) Switch to a larger embedding model

**Q6. (Enterprise RAG)** A multi-tenant SaaS assistant stores every customer's documents in one vector index. During testing, one tenant's answer cited another tenant's document. The current design retrieves the top 20 and then removes chunks from other tenants, and the system prompt says "never reveal other customers' data". What is the right fix?

- A) Apply the tenant (and access-group) filter inside the search, before ranking, or give each tenant its own namespace; treat the prompt rule as defense in depth only
- B) Strengthen the system prompt with an explicit list of forbidden tenant IDs
- C) Retrieve the top 100 instead of the top 20 so enough same-tenant chunks remain after the filter
- D) Ask the model to check the `tenant` field of every chunk before answering

**Q7. (Solution design)** A startup with two engineers already builds on the OpenAI Responses API. It needs a support bot over 300 product PDFs within two weeks, with no special access rules, and wants to prove value before investing more. What do you recommend first?

- A) A custom pipeline with self-hosted pgvector, a separate BM25 service, a reranker and contextual retrieval
- B) Fine-tuning a model on the PDFs
- C) Pasting all 300 PDFs into every prompt
- D) OpenAI vector stores with the `file_search` tool, file attributes for basic filtering, and an eval of retrieved results; move parts to a custom pipeline only if evals show a specific limit

**Q8. (Enterprise RAG)** The team wants to switch from `text-embedding-3-small` to `voyage-4` for better retrieval. The index holds 2 million chunks. A developer proposes embedding only new documents with `voyage-4` and keeping the old vectors. What do you advise?

- A) Agree: cosine similarity works across any two embedding models
- B) Re-embed the whole corpus into a new index, evaluate it against the current one on labeled questions, then switch traffic (keeping the old index for rollback)
- C) Keep both models and average each query's two embeddings
- D) Normalize the old vectors again so they match the new model's space

In [ ]:
answers = {
    "q1": None,
    "q2": None,
    "q3": None,
    "q4": None,
    "q5": None,
    "q6": None,
    "q7": None,
    "q8": None,
}

In [ ]:
ANSWER_HASHES = {'q1': '227effd0f0211c628c0ae190f31feffeed3e6aa4b570e9bd60b91ac5bb2bb16e', 'q2': '3d9c97ee45947882f5eee5cf878c4f33a0a270d8f2a2e1fc259d66cdc95c7b45', 'q3': '14099ffd7333e442b90669f0e435e505e0e45a0ed148540caca94158c15a27c6', 'q4': '61f643a360ded8f53a0309d76dab6c7178f5a30db61b94cdcd57e7341a5aeaaa', 'q5': 'e3c6d5ddd445920945724cd165c914deaf97d5041f947ba2df35e569269b353f', 'q6': 'ac78c6455837b3b074080e7912c1bdab72fbbfdcaeefd9269fe293a50c4a9522', 'q7': '45ff49923feb785985244958470606d87fb75c9912783e470a079e3bb46e1457', 'q8': '39f95f59afaa9522616af26bd34df1e0184030ae8f738e319fc9a1f15b075834'}


def grade_quiz(answers: dict) -> int:
    correct = 0
    for qid in sorted(ANSWER_HASHES, key=lambda q: int(q[1:])):
        given = answers.get(qid)
        letter = given.strip().upper() if isinstance(given, str) else None
        ok = letter is not None and hashlib.sha256(f"{qid}:{letter}".encode()).hexdigest() == ANSWER_HASHES[qid]
        correct += ok
        print(f"{'✓' if ok else '✗'} {qid}: {letter or '(no answer)'}")
    SCORE["quiz"] = correct
    print(f"Quiz: {correct}/{len(ANSWER_HASHES)} = {correct / len(ANSWER_HASHES):.0%}")
    return correct


_ = grade_quiz(answers)

## Part B — Coding exercises (offline)

Each exercise: read the task, replace the stub, run the checker.

### Exercise 1 — Word windows with overlap

Write `chunk_words(text, max_words, overlap)` that splits `text` on whitespace and returns a list of strings (words joined by single spaces):

- each window has at most `max_words` words; window *i* starts at word `i * (max_words - overlap)`;
- stop after the first window that reaches the last word (no window that only repeats the end of the previous one);
- empty or whitespace-only text returns `[]`;
- raise `ValueError` if `max_words < 1`, `overlap < 0` or `overlap >= max_words`.

Theory: [6.3](README.md#63-chunking-strategies)

In [ ]:
def chunk_words(text: str, max_words: int, overlap: int) -> list[str]:
    """Fixed-size word windows with overlap (rules in the task above)."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex1() -> None:
    def check():
        text = " ".join(f"w{i}" for i in range(1, 11))          # w1 ... w10
        got = chunk_words(text, 4, 1)
        assert isinstance(got, list), f"return a list of strings, got {type(got).__name__}"
        assert got == ["w1 w2 w3 w4", "w4 w5 w6 w7", "w7 w8 w9 w10"], f"chunk_words(10 words, 4, 1) = {got}"
        got = chunk_words(text, 4, 0)
        assert got == ["w1 w2 w3 w4", "w5 w6 w7 w8", "w9 w10"], f"overlap=0 gives {got}"
        got = chunk_words(" ".join(f"w{i}" for i in range(1, 9)), 4, 2)
        assert got == ["w1 w2 w3 w4", "w3 w4 w5 w6", "w5 w6 w7 w8"], (
            f"8 words, 4/2 gives {got}: stop once a window reaches the last word")
        assert chunk_words("  one\n two  ", 10, 2) == ["one two"], "split on any whitespace, join with single spaces"
        assert chunk_words("   ", 5, 1) == [], "empty text -> []"
        for bad in [(3, 3), (3, 5), (0, 0), (3, -1)]:
            try:
                chunk_words(text, *bad)
            except ValueError:
                continue
            raise AssertionError(f"chunk_words(text, {bad[0]}, {bad[1]}) must raise ValueError")
    _run_check(1, check)


check_ex1()

### Exercise 2 — Reciprocal rank fusion

Write `rrf(rankings, k=60, weights=None)`. `rankings` is a list of ranked lists of chunk ids (best first). Each list contributes `weight / (k + rank)` to every id in it, with `rank` starting at **1**; `weights` defaults to 1.0 for every list. Return all ids sorted by fused score, highest first. Ties keep the order in which ids first appear (scan the lists in order); a stable sort over a dict filled in that order does this.

Theory: [6.7](README.md#67-hybrid-search-a-multi-index-pipeline-with-rank-fusion)

In [ ]:
def rrf(rankings: list[list[str]], k: int = 60, weights: list[float] | None = None) -> list[str]:
    """Fuse ranked lists of ids by reciprocal rank (rules in the task above)."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex2() -> None:
    def check():
        dense, lexical = ["a", "b", "c", "d"], ["c", "e", "a"]
        got = rrf([dense, lexical])
        assert isinstance(got, list), "return a list of ids"
        assert set(got) == {"a", "b", "c", "d", "e"}, f"every id from every list must appear once, got {got}"
        assert got[:2] == ["a", "c"], f"a (ranks 1 and 3) and c (3 and 1) tie at the top, a first; got {got}"
        assert got == ["a", "c", "b", "e", "d"], f"expected ['a', 'c', 'b', 'e', 'd'], got {got}"
        assert rrf([dense, lexical], k=1)[:2] == ["a", "c"], "k changes the scores, not this tie"
        got = rrf([dense, lexical], weights=[1.0, 3.0])
        assert got[0] == "c" and got.index("e") < got.index("b"), f"weights=[1, 3] should favor the second list, got {got}"
        assert rrf([["x"], []]) == ["x"], "an empty ranking contributes nothing"
    _run_check(2, check)


check_ex2()

### Exercise 3 — Recall@k and MRR

Write two metrics over a labeled question set. `results` maps a question id to its ranked list of chunk ids; `gold` maps the same question ids to the id of the chunk that answers it.

- `recall_at_k(results, gold, k)`: the share of questions in `gold` whose gold chunk is among the first `k` results (a question missing from `results` counts as a miss).
- `mrr(results, gold)`: the mean over questions in `gold` of `1 / rank` of the gold chunk (rank starts at 1), counting 0 when it is not retrieved.
- Both return `0.0` when `gold` is empty.

Theory: [6.6](README.md#66-bm25-lexical-search) (the practice notebook's `evaluate()`)

In [ ]:
def recall_at_k(results: dict[str, list[str]], gold: dict[str, str], k: int) -> float:
    # TODO
    raise NotImplementedError


def mrr(results: dict[str, list[str]], gold: dict[str, str]) -> float:
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex3() -> None:
    def check():
        gold = {"q1": "a", "q2": "b", "q3": "c", "q4": "d"}
        results = {"q1": ["a", "x"], "q2": ["x", "y", "b"], "q3": ["x", "y", "z", "w"]}   # q4 missing
        assert recall_at_k(results, gold, 1) == 0.25, f"recall@1 = {recall_at_k(results, gold, 1)}, expected 0.25"
        assert recall_at_k(results, gold, 3) == 0.5, f"recall@3 = {recall_at_k(results, gold, 3)}, expected 0.5"
        assert recall_at_k(results, gold, 10) == 0.5, "a question missing from results is a miss"
        got = mrr(results, gold)
        assert abs(got - (1 + 1 / 3) / 4) < 1e-9, f"mrr = {got}, expected {(1 + 1 / 3) / 4:.4f}"
        assert recall_at_k({}, {}, 5) == 0.0 and mrr({}, {}) == 0.0, "empty gold -> 0.0"
    _run_check(3, check)


check_ex3()

### Exercise 4 — A cited Claude request from retrieved chunks

Write `build_cited_request(question, hits)` that returns the **keyword arguments** for `client.messages.create(**kwargs)`, or `None` when `hits` is empty (the "not covered" path: no model call). The kwargs must contain:

- `model=MODEL`, `max_tokens` between 1 and 2048, `system=GROUNDED`;
- exactly one user message whose `content` is one `search_result` block per hit, **in the order of `hits`**, followed by one text block with the question;
- each `search_result` block: `"source": hit.meta["source"]`, `"title": hit.meta["title"]`, `"content": [{"type": "text", "text": hit.text}]` and `"citations": {"enabled": True}`.

The checker sends your kwargs through a real `anthropic` client with a fake HTTP layer.

Theory: [6.11](README.md#611-grounded-generation-with-citations)

In [ ]:
MODEL = "claude-sonnet-5-5"
GROUNDED = ("Answer only from the provided sources. If they do not contain the answer, say you cannot find it "
            "in the documents. Never use general knowledge.")

In [ ]:
def build_cited_request(question: str, hits: list[Chunk]) -> dict | None:
    """messages.create kwargs with one search_result block per hit; None when there are no hits."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex4() -> None:
    def check():
        hits = [Chunk("ember#e-305", "ember", "E-305 means the fuel valve is blocked.",
                      {"source": "kb://ember?v=1", "title": "Ember stove — Error E-305"}),
                Chunk("ember#safety", "ember", "Never use the stove inside a tent.",
                      {"source": "kb://ember?v=1#safety", "title": "Ember stove — Safety"})]
        assert build_cited_request("anything?", []) is None, "no hits -> return None (do not call the model)"
        kwargs = build_cited_request("What does E-305 mean?", hits)
        assert isinstance(kwargs, dict), "return a dict of messages.create kwargs"
        assert kwargs.get("model") == MODEL and 0 < kwargs.get("max_tokens", 0) <= 2048, "model=MODEL, max_tokens 1..2048"
        assert kwargs.get("system") == GROUNDED, "system=GROUNDED"
        msgs = kwargs.get("messages", [])
        assert len(msgs) == 1 and msgs[0].get("role") == "user", "exactly one user message"
        content = msgs[0]["content"]
        assert isinstance(content, list) and len(content) == 3, "content = one search_result per hit + one text block"
        for block, h in zip(content[:2], hits):
            assert block.get("type") == "search_result", "the first blocks must be search_result blocks, in hit order"
            assert block.get("source") == h.meta["source"] and block.get("title") == h.meta["title"], "source/title from meta"
            assert block.get("content") == [{"type": "text", "text": h.text}], "content: one text block with the chunk text"
            assert block.get("citations") == {"enabled": True}, "enable citations on every search_result"
        assert content[2] == {"type": "text", "text": "What does E-305 mean?"}, "the question goes last, as a text block"
        seen = []
        fake_claude(seen).messages.create(**kwargs)
        assert seen and seen[0]["messages"][0]["content"][0]["type"] == "search_result", "the SDK must send it unchanged"
    _run_check(4, check)


check_ex4()

### Exercise 5 — Cache-friendly contextualizer requests

Write `build_context_requests(doc_text, chunks)` that returns one `messages.create` kwargs dict per chunk, in order:

- `model="claude-haiku-4-5"`, `max_tokens` between 1 and 200, no `system`;
- one user message with exactly **two** text blocks: first `f"<document>{doc_text}</document>"` with `"cache_control": {"type": "ephemeral"}`, then `CONTEXT_PROMPT.format(chunk=chunk.text)` **without** `cache_control`.

Why it matters: the cache matches on an identical prefix. If the document block changes between requests (or comes after the chunk), every request pays for the whole document again.

Theory: [6.10](README.md#610-contextual-retrieval)

In [ ]:
CONTEXT_PROMPT = ("Here is a chunk from the document above:\n<chunk>{chunk}</chunk>\nWrite one or two sentences that "
                  "situate this chunk within the document, to improve search retrieval. "
                  "Answer with only those sentences.")

In [ ]:
def build_context_requests(doc_text: str, chunks: list[Chunk]) -> list[dict]:
    """One kwargs dict per chunk; the document block is the identical, cached prefix."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex5() -> None:
    def check():
        doc = "# Summit 40 backpack owner's guide\n## Warranty\nThe frame sheet is covered for 2 years.\n## Care\nHand wash."
        chunks = [Chunk("s#w", "summit-40", "The frame sheet is covered for 2 years."), Chunk("s#c", "summit-40", "Hand wash.")]
        reqs = build_context_requests(doc, chunks)
        assert isinstance(reqs, list) and len(reqs) == 2, "return one kwargs dict per chunk"
        prefixes = []
        for req, c in zip(reqs, chunks):
            assert req.get("model") == "claude-haiku-4-5", "use the fast model claude-haiku-4-5"
            assert 0 < req.get("max_tokens", 0) <= 200, "max_tokens between 1 and 200 (the context is 50-100 tokens)"
            assert "system" not in req, "no system prompt in this exercise"
            msgs = req.get("messages", [])
            assert len(msgs) == 1 and msgs[0]["role"] == "user" and len(msgs[0]["content"]) == 2, "one user message, two blocks"
            first, second = msgs[0]["content"]
            assert first.get("text") == f"<document>{doc}</document>", "block 1 is the whole document in <document> tags"
            assert first.get("cache_control") == {"type": "ephemeral"}, "mark the document block with cache_control"
            assert second.get("text") == CONTEXT_PROMPT.format(chunk=c.text), "block 2 is CONTEXT_PROMPT with this chunk"
            assert "cache_control" not in second, "do not cache the per-chunk block: it differs every time"
            prefixes.append(json.dumps(first, sort_keys=True))
        assert len(set(prefixes)) == 1, "the cached document block must be byte-identical across requests"
        seen = []
        client = fake_claude(seen)
        for req in reqs:
            client.messages.create(**req)
        assert len(seen) == 2, "the SDK should accept every request"
    _run_check(5, check)


check_ex5()

## Part C — Architecture scenario

**City water utility: resident and staff assistant.** A city water utility wants one assistant for two audiences. **Residents** ask about bills, tariffs, outages and moving house, on the website, in English and Spanish. **Field and office staff** ask about internal procedures and equipment manuals. The corpus: 2,000 tariff and policy PDFs (many with tables; some scanned), 30,000 pages of equipment manuals full of part numbers and fault codes, and outage notices that change every hour. Internal procedures must never reach residents. The city's IT team runs Postgres on Azure and has budget for one engineer. Leadership wants a recommendation on Claude vs OpenAI and on managed File Search vs a custom pipeline.

Describe your design (15–25 lines):
1. Ingestion: parsing (tables, scans), cleaning, metadata, and how hourly outage notices stay fresh.
2. Chunking and indexing: strategy per content type, number and kind of indexes, partitioning.
3. Retrieval: hybrid search, reranking, query rewriting (two languages), contextual retrieval: which, and why.
4. Access control: where the resident/staff boundary is enforced.
5. Generation: citations and the "not covered" path, on Claude and on OpenAI.
6. Managed vs custom, and Claude vs OpenAI: your recommendation and what would change it.

*Your answer here.*

<details><summary>Rubric: what a strong answer contains</summary>

- **Ingestion per content type.** A layout-aware PDF parser plus OCR for scanned pages; tables kept as rows with their headers (a tariff cell means nothing without its column). Metadata on every record: `source_id`, title, section path, `version`, `effective_at`, `language`, `audience` (`resident`/`staff`), `content_hash`. Outage notices handled as a **separate, small, frequently refreshed index** (or a live API/tool), not mixed into the monthly PDF re-index; superseded notices marked `is_current=False` or expired.
- **Chunking.** Structure-aware chunks for policies and manuals (sections, procedures, one fault code per chunk where the manual allows), title and section prefixed; tables chunked by logical row groups. Not one fixed size for everything.
- **Indexes.** Dense + BM25 for every content type (fault codes and part numbers need BM25), fused with RRF; pgvector plus Postgres full-text search (or a BM25 extension) fits the IT team's stack. A multilingual embedding model (or translate the query) so Spanish questions find English documents, or keep per-language indexes.
- **Retrieval.** Rewrite follow-ups to standalone queries; rerank the fused top 50–100 to the top 5–10; contextual retrieval for the manuals (many similar sections across models), with prompt caching and batch processing, re-run only for changed documents. Justified by an eval on labeled questions, not assumed.
- **Access control.** The resident/staff boundary is a **filter inside the search** (`audience`), or separate indexes per audience, driven by the authenticated identity. Never a prompt instruction, never a post-filter after top-k. Residents' requests can only reach the public index.
- **Generation.** Answer only from sources; cite every claim (Claude: `search_result` blocks and `search_result_location` citations; OpenAI: `file_citation` annotations with `file_search`, or `[S1]` markers checked in code with custom retrieval). A "not covered" gate on the reranker score plus a hand-off to a human or the outage map; no uncited policy or billing answers.
- **Recommendation.** A custom pipeline in the city's Postgres is the strong default here: hard audience separation, hourly freshness, code-heavy manuals, one store the IT team already runs. Managed `file_search` is reasonable for a quick resident-only prototype if data may live in OpenAI's store. Either vendor works for generation behind one interface; the choice turns on existing contracts and cloud (Azure: OpenAI on Azure or Claude in Microsoft Foundry), data-residency terms, and eval results on the city's own questions. Names what evidence would change the recommendation.
- **Measurement.** Mentions a labeled question set (both languages, both audiences) with recall@k and answer checks before launch; detailed evaluation is chapter 07's topic.
</details>

## Scorecard

In [ ]:
quiz = SCORE.get("quiz", 0)
n_quiz, n_ex = 8, 5
passed = sum(RESULTS.get(i, False) for i in range(1, n_ex + 1))
overall = (quiz + passed) / (n_quiz + n_ex)
print(f"Quiz:       {quiz}/{n_quiz} ({quiz / n_quiz:.0%})")
print(f"Exercises:  {passed}/{n_ex} passed  " + " ".join(f"{i}:{'✓' if RESULTS.get(i) else '✗'}" for i in range(1, n_ex + 1)))
print(f"Overall:    {overall:.0%}  (pass mark 72%)")
print("PASS: on to chapter 07" if overall >= 0.72 else "RETRY: reread the README sections behind your ✗ items, then try again")

Next: chapter 07 — Retrieval strategy and knowledge operations (planned). Until it exists, go back to the [track overview](../README.md).

---

## Solutions (spoilers)

Open these only after you have attempted everything.

<details><summary>Quiz answers with explanations</summary>

**Q1:** **C.** A stable 70,000-token corpus that every employee may read fits in the context window, and prompt caching makes repeated questions cheap. RAG adds moving parts without a benefit here. A is false as a rule. B does not give reliable facts or citations, and every handbook change would need retraining. D throws away context and invents a retrieval problem.

**Q2:** **A.** Clauses are the natural unit for both retrieval and citation, and the heading path makes every citation precise. B still cuts clauses, only less often, and makes chunks less focused. C asks the model to fix a chunking problem it cannot see. D does not move the boundaries.

**Q3:** **D.** A very low top score is a reliable sign that the documents do not cover the question. Gating in code skips a model call that could only guess, and the grounded-only instruction catches what the gate misses. A adds more unrelated chunks. B is a prompt rule against a retrieval gap: the model still receives an unrelated chunk and may stretch it to fit. C answers more fluently from the same wrong chunk.

**Q4:** **B.** Caching the document prefix means chunks 2–12 pay the cache-read price for it; a fast model and the Batch API's discount cut the rest; incremental runs avoid paying twice. A raises cost. C removes the document, which is the whole point. D throws away part of the measured gain (contextual BM25 adds to contextual embeddings).

**Q5:** **C.** A compound question asks one query to find two things, and one side wins the ranking. Decomposition gives each plan its own retrieval, and the generator then compares the two result sets. A makes every chunk less focused and still does not guarantee both plans share a chunk. B helps when the question is phrased very differently from the documents, not when two topics compete. D does not change the fact that one query must find two separate sections.

**Q6:** **A.** Access rules belong inside the search (or in physically separate namespaces), so forbidden chunks never compete for the top k and never reach the model. B and D rely on the model to enforce security. C only makes post-filtering waste less often; the leak path remains.

**Q7:** **D.** Managed `file_search` matches the stack, the deadline and the team size, and attributes cover basic filtering. Evals then show whether any stage needs custom work. A is too much engineering before value is proven. B gives no citations or freshness. C does not fit and costs too much.

**Q8:** **B.** Vectors from different models live in different spaces and cannot be compared, so the corpus must be re-embedded; a blue/green swap after an eval keeps a rollback path. A, C and D all mix incompatible vectors.
</details>

<details><summary>Exercise 1 — Word windows with overlap: reference solution</summary>

```python
def chunk_words(text: str, max_words: int, overlap: int) -> list[str]:
    if max_words < 1 or overlap < 0 or overlap >= max_words:
        raise ValueError("need max_words >= 1 and 0 <= overlap < max_words")
    words, step, out = text.split(), max_words - overlap, []
    for start in range(0, len(words), step):
        out.append(" ".join(words[start:start + max_words]))
        if start + max_words >= len(words):      # this window reached the last word
            break
    return out
```
</details>

<details><summary>Exercise 2 — Reciprocal rank fusion: reference solution</summary>

```python
def rrf(rankings: list[list[str]], k: int = 60, weights: list[float] | None = None) -> list[str]:
    weights = weights or [1.0] * len(rankings)
    scores: dict[str, float] = {}
    for w, ranking in zip(weights, rankings):
        for rank, cid in enumerate(ranking, start=1):
            scores[cid] = scores.get(cid, 0.0) + w / (k + rank)
    return sorted(scores, key=lambda cid: -scores[cid])
```
</details>

<details><summary>Exercise 3 — Recall@k and MRR: reference solution</summary>

```python
def recall_at_k(results: dict[str, list[str]], gold: dict[str, str], k: int) -> float:
    if not gold:
        return 0.0
    return sum(g in results.get(q, [])[:k] for q, g in gold.items()) / len(gold)


def mrr(results: dict[str, list[str]], gold: dict[str, str]) -> float:
    if not gold:
        return 0.0
    total = 0.0
    for q, g in gold.items():
        ranked = results.get(q, [])
        total += 1 / (ranked.index(g) + 1) if g in ranked else 0.0
    return total / len(gold)
```
</details>

<details><summary>Exercise 4 — A cited Claude request from retrieved chunks: reference solution</summary>

```python
def build_cited_request(question: str, hits: list[Chunk]) -> dict | None:
    if not hits:
        return None
    blocks = [{"type": "search_result", "source": h.meta["source"], "title": h.meta["title"],
               "content": [{"type": "text", "text": h.text}], "citations": {"enabled": True}} for h in hits]
    return dict(model=MODEL, max_tokens=1024, system=GROUNDED,
                messages=[{"role": "user", "content": blocks + [{"type": "text", "text": question}]}])
```
</details>

<details><summary>Exercise 5 — Cache-friendly contextualizer requests: reference solution</summary>

```python
def build_context_requests(doc_text: str, chunks: list[Chunk]) -> list[dict]:
    doc_block = {"type": "text", "text": f"<document>{doc_text}</document>", "cache_control": {"type": "ephemeral"}}
    return [dict(model="claude-haiku-4-5", max_tokens=150,
                 messages=[{"role": "user", "content": [
                     dict(doc_block), {"type": "text", "text": CONTEXT_PROMPT.format(chunk=c.text)}]}])
            for c in chunks]
```
</details>